# Modeling and Evaluation

## Stage 9: Train-Test Split and Preprocessing Pipeline

This stage prepares the dataset for machine-learning models.

The data is divided into stratified training and test sets using an
80/20 ratio. Rule-based corrections are applied before splitting, while
all learned preprocessing operations are fitted only on the training
data.

Two preprocessing configurations are created: one including the dataset
source feature and another excluding it.

In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    StandardScaler,
)


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)


CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data.csv"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


print("Project root:")
print(PROJECT_ROOT)

print("\nData path:")
print(DATA_PATH)

print("\nData file exists:")
print(DATA_PATH.exists())

Project root:
E:\Projects\DataMining

Data path:
E:\Projects\DataMining\data\raw\data.csv

Data file exists:
True


In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The dataset file was not found.\n"
        f"Checked path: {DATA_PATH}"
    )


df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()


print("Dataset loaded successfully.")
print(f"Number of rows: {df_raw.shape[0]}")
print(f"Number of columns: {df_raw.shape[1]}")

Dataset loaded successfully.
Number of rows: 920
Number of columns: 16


In [4]:
ID_COLUMN = "id"
TARGET_COLUMN = "num"


NUMERIC_FEATURES = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak",
]


CATEGORICAL_FEATURES_WITH_DATASET = [
    "sex",
    "dataset",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
]


CATEGORICAL_FEATURES_WITHOUT_DATASET = [
    feature
    for feature in CATEGORICAL_FEATURES_WITH_DATASET
    if feature != "dataset"
]


FEATURES_WITH_DATASET = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES_WITH_DATASET
)


FEATURES_WITHOUT_DATASET = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES_WITHOUT_DATASET
)


required_columns = (
    [ID_COLUMN, TARGET_COLUMN]
    + FEATURES_WITH_DATASET
)


missing_columns = [
    column
    for column in required_columns
    if column not in df_raw.columns
]


if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )


print(f"Numeric features: {len(NUMERIC_FEATURES)}")
print(
    "Categorical features with dataset: "
    f"{len(CATEGORICAL_FEATURES_WITH_DATASET)}"
)
print(
    "Categorical features without dataset: "
    f"{len(CATEGORICAL_FEATURES_WITHOUT_DATASET)}"
)

Numeric features: 5
Categorical features with dataset: 9
Categorical features without dataset: 8


In [5]:
X_model = df_raw.drop(
    columns=[
        ID_COLUMN,
        TARGET_COLUMN,
    ]
).copy(deep=True)


y = df_raw[TARGET_COLUMN].astype(int).copy()


SUSPICIOUS_ZERO_FEATURES = [
    "trestbps",
    "chol",
]


zero_replacement_rows = []


for feature in SUSPICIOUS_ZERO_FEATURES:
    zero_count = int(
        (X_model[feature] == 0).sum()
    )

    original_missing_count = int(
        X_model[feature].isna().sum()
    )

    X_model[feature] = X_model[feature].replace(
        0,
        np.nan,
    )

    adjusted_missing_count = int(
        X_model[feature].isna().sum()
    )

    zero_replacement_rows.append(
        {
            "feature": feature,
            "zero_values_replaced": zero_count,
            "original_missing_count": original_missing_count,
            "adjusted_missing_count": adjusted_missing_count,
        }
    )


zero_replacement_report = pd.DataFrame(
    zero_replacement_rows
)


display(zero_replacement_report)

,feature,zero_values_replaced,original_missing_count,adjusted_missing_count
0,trestbps,1,59,60
1,chol,172,30,202


In [6]:
def normalize_category_value(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(int(numeric_value))

    return str(value)


for feature in CATEGORICAL_FEATURES_WITH_DATASET:
    X_model[feature] = X_model[feature].map(
        normalize_category_value
    )


categorical_type_summary = pd.DataFrame(
    {
        "feature": CATEGORICAL_FEATURES_WITH_DATASET,
        "data_type": [
            str(X_model[feature].dtype)
            for feature in CATEGORICAL_FEATURES_WITH_DATASET
        ],
        "missing_count": [
            int(X_model[feature].isna().sum())
            for feature in CATEGORICAL_FEATURES_WITH_DATASET
        ],
        "unique_non_missing_values": [
            int(
                X_model[feature].nunique(
                    dropna=True
                )
            )
            for feature in CATEGORICAL_FEATURES_WITH_DATASET
        ],
    }
)


display(categorical_type_summary)

,feature,data_type,missing_count,unique_non_missing_values
0,sex,object,0,2
1,dataset,object,0,4
2,cp,object,0,4
3,fbs,object,90,2
4,restecg,object,2,3
5,exang,object,55,2
6,slope,object,309,3
7,ca,object,611,4
8,thal,object,486,3


In [7]:
RANDOM_STATE = 42
TEST_SIZE = 0.20


X_train, X_test, y_train, y_test = (
    train_test_split(
        X_model,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )
)


print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

print(
    "\nTraining percentage: "
    f"{len(X_train) / len(X_model) * 100:.1f}%"
)

print(
    "Test percentage: "
    f"{len(X_test) / len(X_model) * 100:.1f}%"
)

Training rows: 736
Test rows: 184

Training percentage: 80.0%
Test percentage: 20.0%


In [8]:
def build_target_distribution(target_values, split_name):
    distribution = (
        target_values
        .value_counts()
        .sort_index()
        .rename_axis("target_class")
        .reset_index(name="count")
    )

    distribution["percent"] = (
        distribution["count"]
        / len(target_values)
        * 100
    )

    distribution.insert(
        0,
        "split",
        split_name,
    )

    return distribution


train_target_distribution = (
    build_target_distribution(
        target_values=y_train,
        split_name="train",
    )
)


test_target_distribution = (
    build_target_distribution(
        target_values=y_test,
        split_name="test",
    )
)


print("Training target distribution:")
display(train_target_distribution.round(2))


print("Test target distribution:")
display(test_target_distribution.round(2))

Training target distribution:


,split,target_class,count,percent
0,train,0,329,44.70
1,train,1,212,28.80
2,train,2,87,11.82
3,train,3,86,11.68
4,train,4,22,2.99


Test target distribution:


,split,target_class,count,percent
0,test,0,82,44.57
1,test,1,53,28.80
2,test,2,22,11.96
3,test,3,21,11.41
4,test,4,6,3.26


In [9]:
train_assignments = pd.DataFrame(
    {
        "row_index": X_train.index,
        "record_id": df_raw.loc[
            X_train.index,
            ID_COLUMN,
        ].to_numpy(),
        "target_class": y_train.to_numpy(),
        "split": "train",
    }
)


test_assignments = pd.DataFrame(
    {
        "row_index": X_test.index,
        "record_id": df_raw.loc[
            X_test.index,
            ID_COLUMN,
        ].to_numpy(),
        "target_class": y_test.to_numpy(),
        "split": "test",
    }
)


split_assignments = pd.concat(
    [
        train_assignments,
        test_assignments,
    ],
    ignore_index=True,
)


split_assignments = split_assignments.sort_values(
    by="row_index"
).reset_index(drop=True)


display(split_assignments.head())

,row_index,record_id,target_class,split
0,0,1,0,train
1,1,2,2,train
2,2,3,1,test
3,3,4,0,test
4,4,5,0,train


In [10]:
def build_preprocessor(
    numeric_features,
    categorical_features,
    numeric_imputation_strategy="median",
    scaler_name="standard",
):
    valid_imputation_strategies = {
        "mean",
        "median",
    }

    if (
        numeric_imputation_strategy
        not in valid_imputation_strategies
    ):
        raise ValueError(
            "Numeric imputation strategy must be "
            "'mean' or 'median'."
        )

    if scaler_name == "standard":
        scaler = StandardScaler()
    elif scaler_name == "minmax":
        scaler = MinMaxScaler()
    else:
        raise ValueError(
            "Scaler name must be "
            "'standard' or 'minmax'."
        )

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy=numeric_imputation_strategy
                ),
            ),
            (
                "scaler",
                scaler,
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Missing",
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    preprocessor.set_output(
        transform="pandas"
    )

    return preprocessor

In [11]:
preprocessor_with_dataset = build_preprocessor(
    numeric_features=NUMERIC_FEATURES,
    categorical_features=(
        CATEGORICAL_FEATURES_WITH_DATASET
    ),
    numeric_imputation_strategy="median",
    scaler_name="standard",
)


preprocessor_without_dataset = build_preprocessor(
    numeric_features=NUMERIC_FEATURES,
    categorical_features=(
        CATEGORICAL_FEATURES_WITHOUT_DATASET
    ),
    numeric_imputation_strategy="median",
    scaler_name="standard",
)


print("Preprocessors created successfully.")

Preprocessors created successfully.


In [12]:
X_train_with_dataset = X_train[
    FEATURES_WITH_DATASET
].copy()


X_test_with_dataset = X_test[
    FEATURES_WITH_DATASET
].copy()


X_train_without_dataset = X_train[
    FEATURES_WITHOUT_DATASET
].copy()


X_test_without_dataset = X_test[
    FEATURES_WITHOUT_DATASET
].copy()


X_train_transformed_with_dataset = (
    preprocessor_with_dataset.fit_transform(
        X_train_with_dataset
    )
)


X_test_transformed_with_dataset = (
    preprocessor_with_dataset.transform(
        X_test_with_dataset
    )
)


X_train_transformed_without_dataset = (
    preprocessor_without_dataset.fit_transform(
        X_train_without_dataset
    )
)


X_test_transformed_without_dataset = (
    preprocessor_without_dataset.transform(
        X_test_without_dataset
    )
)


print("Preprocessing completed successfully.")

Preprocessing completed successfully.


In [13]:
transformed_shape_summary = pd.DataFrame(
    [
        {
            "configuration": "with_dataset",
            "split": "train",
            "rows": (
                X_train_transformed_with_dataset
                .shape[0]
            ),
            "columns": (
                X_train_transformed_with_dataset
                .shape[1]
            ),
        },
        {
            "configuration": "with_dataset",
            "split": "test",
            "rows": (
                X_test_transformed_with_dataset
                .shape[0]
            ),
            "columns": (
                X_test_transformed_with_dataset
                .shape[1]
            ),
        },
        {
            "configuration": "without_dataset",
            "split": "train",
            "rows": (
                X_train_transformed_without_dataset
                .shape[0]
            ),
            "columns": (
                X_train_transformed_without_dataset
                .shape[1]
            ),
        },
        {
            "configuration": "without_dataset",
            "split": "test",
            "rows": (
                X_test_transformed_without_dataset
                .shape[0]
            ),
            "columns": (
                X_test_transformed_without_dataset
                .shape[1]
            ),
        },
    ]
)


display(transformed_shape_summary)

,configuration,split,rows,columns
0,with_dataset,train,736,38
1,with_dataset,test,184,38
2,without_dataset,train,736,34
3,without_dataset,test,184,34


In [14]:
print("Transformed training data with dataset:")

display(
    X_train_transformed_with_dataset.head()
)


print("Transformed training data without dataset:")

display(
    X_train_transformed_without_dataset.head()
)

Transformed training data with dataset:


,age,trestbps,chol,thalch,oldpeak,sex_Female,sex_Male,dataset_Cleveland,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,cp_asymptomatic,cp_atypical angina,cp_non-anginal,cp_typical angina,fbs_False,fbs_Missing,fbs_True,restecg_Missing,restecg_lv hypertrophy,restecg_normal,restecg_st-t abnormality,exang_False,exang_Missing,exang_True,slope_Missing,slope_downsloping,slope_flat,slope_upsloping,ca_0,ca_1,ca_2,ca_3,ca_Missing,thal_Missing,thal_fixed defect,thal_normal,thal_reversable defect
637,-0.033457,-0.659639,-0.094859,-1.700405,-0.803096,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
743,2.195486,-0.104344,-0.094859,0.084842,-0.329703,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
643,-0.033457,-0.104344,-0.094859,-0.113518,0.143690,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
694,0.921804,-0.937286,-0.094859,-2.612865,-1.276489,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
913,0.921804,1.450481,-1.453267,0.005498,-0.803096,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


Transformed training data without dataset:


,age,trestbps,chol,thalch,oldpeak,sex_Female,sex_Male,cp_asymptomatic,cp_atypical angina,cp_non-anginal,cp_typical angina,fbs_False,fbs_Missing,fbs_True,restecg_Missing,restecg_lv hypertrophy,restecg_normal,restecg_st-t abnormality,exang_False,exang_Missing,exang_True,slope_Missing,slope_downsloping,slope_flat,slope_upsloping,ca_0,ca_1,ca_2,ca_3,ca_Missing,thal_Missing,thal_fixed defect,thal_normal,thal_reversable defect
637,-0.033457,-0.659639,-0.094859,-1.700405,-0.803096,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
743,2.195486,-0.104344,-0.094859,0.084842,-0.329703,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
643,-0.033457,-0.104344,-0.094859,-0.113518,0.143690,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
694,0.921804,-0.937286,-0.094859,-2.612865,-1.276489,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
913,0.921804,1.450481,-1.453267,0.005498,-0.803096,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0


In [15]:
feature_names_with_dataset = pd.DataFrame(
    {
        "feature_name": (
            X_train_transformed_with_dataset
            .columns
            .tolist()
        ),
        "configuration": "with_dataset",
    }
)


feature_names_without_dataset = pd.DataFrame(
    {
        "feature_name": (
            X_train_transformed_without_dataset
            .columns
            .tolist()
        ),
        "configuration": "without_dataset",
    }
)


print("Feature names with dataset:")
display(feature_names_with_dataset.head(30))


print("Feature names without dataset:")
display(feature_names_without_dataset.head(30))

Feature names with dataset:


,feature_name,configuration
0,age,with_dataset
1,trestbps,with_dataset
2,chol,with_dataset
3,thalch,with_dataset
4,oldpeak,with_dataset
5,sex_Female,with_dataset
6,sex_Male,with_dataset
7,dataset_Cleveland,with_dataset
8,dataset_Hungary,with_dataset
9,dataset_Switzerland,with_dataset


Feature names without dataset:


,feature_name,configuration
0,age,without_dataset
1,trestbps,without_dataset
2,chol,without_dataset
3,thalch,without_dataset
4,oldpeak,without_dataset
5,sex_Female,without_dataset
6,sex_Male,without_dataset
7,cp_asymptomatic,without_dataset
8,cp_atypical angina,without_dataset
9,cp_non-anginal,without_dataset


In [16]:
feature_configuration = pd.DataFrame(
    [
        {
            "configuration": "with_dataset",
            "numeric_feature_count": len(
                NUMERIC_FEATURES
            ),
            "categorical_feature_count": len(
                CATEGORICAL_FEATURES_WITH_DATASET
            ),
            "dataset_feature_included": True,
        },
        {
            "configuration": "without_dataset",
            "numeric_feature_count": len(
                NUMERIC_FEATURES
            ),
            "categorical_feature_count": len(
                CATEGORICAL_FEATURES_WITHOUT_DATASET
            ),
            "dataset_feature_included": False,
        },
    ]
)


preprocessing_configuration = pd.DataFrame(
    [
        {
            "component": "Suspicious zeros",
            "method": "Replace with missing",
            "fit_scope": "Rule based",
        },
        {
            "component": "Numeric imputation",
            "method": "Median",
            "fit_scope": "Training data only",
        },
        {
            "component": "Categorical imputation",
            "method": "Constant Missing category",
            "fit_scope": "Training data only",
        },
        {
            "component": "Categorical encoding",
            "method": "One-Hot Encoding",
            "fit_scope": "Training data only",
        },
        {
            "component": "Numeric scaling",
            "method": "StandardScaler",
            "fit_scope": "Training data only",
        },
        {
            "component": "Train-test split",
            "method": "Stratified 80/20 split",
            "fit_scope": "Random state 42",
        },
    ]
)


display(feature_configuration)
display(preprocessing_configuration)

,configuration,numeric_feature_count,categorical_feature_count,dataset_feature_included
0,with_dataset,5,9,True
1,without_dataset,5,8,False


,component,method,fit_scope
0,Suspicious zeros,Replace with missing,Rule based
1,Numeric imputation,Median,Training data only
2,Categorical imputation,Constant Missing category,Training data only
3,Categorical encoding,One-Hot Encoding,Training data only
4,Numeric scaling,StandardScaler,Training data only
5,Train-test split,Stratified 80/20 split,Random state 42


In [17]:
zero_replacement_report.to_csv(
    TABLES_DIR
    / "09_zero_replacement_report.csv",
    index=False,
    encoding="utf-8-sig",
)


train_target_distribution.to_csv(
    TABLES_DIR
    / "09_train_target_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)


test_target_distribution.to_csv(
    TABLES_DIR
    / "09_test_target_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)


split_assignments.to_csv(
    TABLES_DIR
    / "09_split_assignments.csv",
    index=False,
    encoding="utf-8-sig",
)


feature_configuration.to_csv(
    TABLES_DIR
    / "09_feature_configuration.csv",
    index=False,
    encoding="utf-8-sig",
)


preprocessing_configuration.to_csv(
    TABLES_DIR
    / "09_preprocessing_configuration.csv",
    index=False,
    encoding="utf-8-sig",
)


transformed_shape_summary.to_csv(
    TABLES_DIR
    / "09_transformed_shape_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


feature_names_with_dataset.to_csv(
    TABLES_DIR
    / "09_feature_names_with_dataset.csv",
    index=False,
    encoding="utf-8-sig",
)


feature_names_without_dataset.to_csv(
    TABLES_DIR
    / "09_feature_names_without_dataset.csv",
    index=False,
    encoding="utf-8-sig",
)


print("Stage 9 outputs saved successfully.")

Stage 9 outputs saved successfully.


In [18]:
assert df_raw.shape == (920, 16), (
    "The raw dataset shape changed unexpectedly."
)

assert len(X_train) == 736, (
    "The training-set size is incorrect."
)

assert len(X_test) == 184, (
    "The test-set size is incorrect."
)

assert set(X_train.index).isdisjoint(
    set(X_test.index)
), (
    "Training and test indices overlap."
)

assert sorted(
    y_train.unique().tolist()
) == [0, 1, 2, 3, 4], (
    "The training set does not contain all target classes."
)

assert sorted(
    y_test.unique().tolist()
) == [0, 1, 2, 3, 4], (
    "The test set does not contain all target classes."
)

assert not (
    X_train_transformed_with_dataset
    .isna()
    .any()
    .any()
), (
    "Missing values remain in the transformed training data."
)

assert not (
    X_test_transformed_with_dataset
    .isna()
    .any()
    .any()
), (
    "Missing values remain in the transformed test data."
)

assert (
    X_train_transformed_with_dataset
    .columns
    .tolist()
    ==
    X_test_transformed_with_dataset
    .columns
    .tolist()
), (
    "Training and test feature columns do not match."
)

assert (
    X_train_transformed_without_dataset
    .columns
    .tolist()
    ==
    X_test_transformed_without_dataset
    .columns
    .tolist()
), (
    "Training and test feature columns do not match."
)

assert (
    X_train_transformed_with_dataset.shape[1]
    >
    X_train_transformed_without_dataset.shape[1]
), (
    "The dataset feature did not increase the encoded feature count."
)

assert split_assignments["row_index"].is_unique, (
    "Split assignment row indices must be unique."
)

assert len(split_assignments) == len(df_raw), (
    "Split assignments do not cover every dataset row."
)


required_output_files = [
    TABLES_DIR / "09_zero_replacement_report.csv",
    TABLES_DIR / "09_train_target_distribution.csv",
    TABLES_DIR / "09_test_target_distribution.csv",
    TABLES_DIR / "09_split_assignments.csv",
    TABLES_DIR / "09_feature_configuration.csv",
    TABLES_DIR / "09_preprocessing_configuration.csv",
    TABLES_DIR / "09_transformed_shape_summary.csv",
    TABLES_DIR / "09_feature_names_with_dataset.csv",
    TABLES_DIR / "09_feature_names_without_dataset.csv",
]


for output_file in required_output_files:
    assert output_file.exists(), (
        f"Expected output file was not created: {output_file}"
    )


print("All Stage 9 validation checks passed.")

All Stage 9 validation checks passed.
